# Gold — dim_keyword_scd1 (SCD 1, poc method)

One row per normalized search keyword (`normalize_keyword`, diacritics kept).
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

**Changed keys** = keywords searched in the window (`stg_search_results.fetched_at`); `keyword_raw_first`, `raw_variant_count`, `first_searched_at` are recomputed over **all** results of those keywords. `dish` / `city` stay NULL as in `dim_keyword`. No `modified_at` → `order_col=None`.

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
from pyspark.sql.window import Window
k = (spark.table(f"{SILVER}.stg_search_results")
     .where("search_keyword IS NOT NULL AND trim(search_keyword) <> ''")
     .withColumn("keyword_normalized", F.expr("normalize_keyword(search_keyword)")))
keys = None if FULL else changed_keys(
    k.where(in_window(vn_date("fetched_at"), FROM_DATE, TO_DATE)).select("keyword_normalized"))
if keys is not None:
    k = k.join(keys, "keyword_normalized", "left_semi")
w = Window.partitionBy("keyword_normalized")
src = (k.withColumn("rn", F.row_number().over(w.orderBy("fetched_at", "search_keyword")))
        .withColumn("raw_variant_count", F.size(F.collect_set("search_keyword").over(w)))
        .withColumn("first_searched_at", F.min("fetched_at").over(w))
        .where("rn = 1")
        .select(F.xxhash64("keyword_normalized").alias("keyword_key"), "keyword_normalized",
                F.expr("unaccent_vi(keyword_normalized)").alias("keyword_unaccented"),
                F.col("search_keyword").alias("keyword_raw_first"), "raw_variant_count",
                F.lit(None).cast("string").alias("dish"), F.lit(None).cast("string").alias("city"), "first_searched_at")
        .withColumn("etl_date", F.current_timestamp())
        .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["keyword_normalized"], exclude=["keyword_key"])

In [ ]:
load_scd1(src, f"{GOLD}.dim_keyword_scd1", """
        keyword_key BIGINT, keyword_normalized STRING, keyword_unaccented STRING, keyword_raw_first STRING,
        raw_variant_count INT, dish STRING, city STRING, first_searched_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Normalized search keyword dimension, SCD 1 (poc checksum method). keyword_key = xxhash64(keyword_normalized), -1 = Unknown.",
    ["keyword_normalized"], LOAD_TYPE, {"keyword_key": "-1", "etl_date": "current_timestamp()", "etl_by": "'Databricks'"},
    order_col=None)